In [ ]:
import time
import librosa
import matplotlib.pyplot as plt
import numpy as np
import sounddevice as sd

# Configuration
SAMPLE_RATE = 44100  # Sampling rate (Hz)


def record_plot_and_extract_table():
    # 1. User input for recording duration
    try:
        duration = float(
            input("Enter recording duration in seconds (e.g., 10): ")
        )
    except ValueError:
        print("Invalid input. Defaulting to 10 seconds.")
        duration = 10.0

    print(f"\nRecording for {duration} seconds... Get ready to sing!")

    # Countdown
    for i in range(3, 0, -1):
        print(f"Starting in {i}...", end="\r")
        time.sleep(1)

    print("\n[RECORDING NOW...]")

    # 2. Capture microphone recording
    audio_data = sd.rec(
        int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1
    )
    sd.wait()
    print("[RECORDING COMPLETE]\nExtracting pitch contour...")

    audio_signal = audio_data.flatten()

    # 3. Extract Fundamental Pitch Frequency (f0)
    f0, voiced_flag, voiced_probs = librosa.pyin(
        audio_signal,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C6"),
        sr=SAMPLE_RATE,
    )

    times = librosa.times_like(f0, sr=SAMPLE_RATE)

    # 4. FIRST: Plot and display the graph
    print("\nDisplaying plot... Close the plot window when ready to extract the table.")
    plt.figure(figsize=(12, 6))

    plt.plot(times, f0, label="Voice Pitch (f0)", color="#007ACC", linewidth=2.5)

   
    

    plt.ylim(100, 600)
    plt.title("Pitch Contour")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Frequency (Hz)")
    plt.grid(True, which="both", linestyle=":", alpha=0.6)
    plt.legend(loc="upper right")
    plt.tight_layout()
    
    # Execution pauses here until you close the plot window
    plt.show()

    # 5. SECOND: Ask for the interval in the terminal AFTER plot window is closed
    try:
        interval = float(
            input("\nEnter time interval in seconds for data table (e.g., 0.5): ")
        )
        if interval <= 0:
            interval = 0.5
    except ValueError:
        print("Invalid input. Defaulting to 0.5 seconds.")
        interval = 0.5

    # 6. Generate time steps starting from t = 0 and interpolate frequencies
    sampled_times = np.arange(0, duration, interval)
    sampled_f0 = np.interp(sampled_times, times, np.nan_to_num(f0, nan=0.0))

    # 7. Print the formatted Data Table
    print("\n" + "=" * 40)
    print(f"{'Time (s)':<15} | {'Frequency (Hz)':<20}")
    print("=" * 40)

    for t, freq in zip(sampled_times, sampled_f0):
        if freq > 0:
            print(f"{t:<15.2f} | {freq:<20.2f}")
        else:
            print(f"{t:<15.2f} | {'Silence / Unvoiced':<20}")

    print("=" * 40 + "\n")


if __name__ == "__main__":
    record_plot_and_extract_table()

Enter recording duration in seconds (e.g., 10):  25



Recording for 25.0 seconds... Get ready to sing!
Starting in 1...
[RECORDING NOW...]
